# Machine Learning for Mineral Exploration
### CU Boulder — Python for Geologists

In this session you will work through three ML techniques that are actually used in mineral exploration:

| Method | Type | What it does |
|--------|------|--------------|
| **PCA** | Unsupervised | Compresses 57 elements into a few summary scores |
| **K-means** | Unsupervised | Groups samples by geochemical similarity — no labels needed |
| **Random Forest** | Supervised | Learns from known deposits to predict new ones |

The data are real stream sediment and soil geochemistry samples from a Nevada gold district, paired with lithology polygons, spectral remote sensing, and airborne geophysics.

**How this notebook works:** Most cells are already written. Cells marked `# --- YOUR TURN ---` ask you to change one or two numbers, re-run the cell, and interpret the output. You cannot break anything — fallback defaults are built in.

---
## Section 0: Setup
*Run these cells once at the start. They install packages (Colab only), import libraries, and point the notebook at the data folder.*

In [ ]:
# SECTION 0: SETUP | ~3 min
# ── Colab only: install missing packages ──────────────────────────────────
import sys, subprocess

COLAB = 'google.colab' in sys.modules
if COLAB:
    print('Installing packages for Colab...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'geopandas', 'rasterio', 'scikit-learn', 'matplotlib', 'numpy', 'pandas', 'scipy'
    ], check=True)
    print('Done.')
else:
    print('Not on Colab — skipping pip install.')

In [ ]:
# Standard imports
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

# Tell the helpers where to find the data
NOTEBOOK_DIR = Path('.')          # adjust if running from a different directory
DATA_DIR     = NOTEBOOK_DIR / 'data'

os.environ['UCB_PROJECT_ROOT'] = str(NOTEBOOK_DIR.resolve())
os.environ['UCB_DATA_ROOT']    = str(DATA_DIR.resolve())

# Load the helper library
import helpers as h

# ── Data paths ─────────────────────────────────────────────────────────────
GEOCHEM_PATH      = DATA_DIR / 'vector' / 'geochem.geojson'
LITHOLOGY_PATH    = DATA_DIR / 'vector' / 'lithology.geojson'
OCCURRENCES_PATH  = DATA_DIR / 'vector' / 'mineral occurrences.geojson'
PROSPECTIVITY_TIF = DATA_DIR / 'ML' / 'stacked_result.tif'

print('Data directory:', DATA_DIR.resolve())
print('Paths configured.')

---
## Section 1: Why ML for Geology?

### The problem

A stream sediment survey in Nevada returns **57 element concentrations** at every sample site. Some of those elements are elevated near gold deposits; most are not. Your job is to find patterns that point toward undiscovered mineralisation.

Human eyes can compare two elements at a time. ML can compare all 57 simultaneously.

### Supervised vs unsupervised

**Unsupervised ML** (PCA, K-means): no labels needed. The algorithm finds structure in the data on its own — like sorting coins by size and colour without knowing what each coin is worth.

**Supervised ML** (Random Forest): you provide examples of what you're looking for — known deposits — and the algorithm learns what makes those sites different from background samples. It then scores every undrilled location.

> **Instructor note:** This distinction is the key conceptual takeaway. Return to it after each section and ask students which type they just ran.

---
## Section 2: Look at the Data
*Before fitting any model, always look at what you have.*

In [ ]:
# SECTION 2: DATA LOOK | ~5 min | Target: min 10

# Load the datasets
geochem_gdf    = h.ensure_xy(gpd.read_file(GEOCHEM_PATH))
vector_gdf     = gpd.read_file(LITHOLOGY_PATH)
targets_gdf    = gpd.read_file(OCCURRENCES_PATH)

# Feature columns = numeric, not coordinates
EXCLUDE = {'X', 'Y', 'id', 'coord_x', 'coord_y', 'elevation_m', 'OBJECTID', 'FID'}
feature_cols = [
    c for c in geochem_gdf.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE
]

print(f"Geochem samples : {len(geochem_gdf)}")
print(f"Lithology units : {len(vector_gdf)}")
print(f"Known occurrences : {len(targets_gdf)}")
print(f"\nGeochem element columns ({len(feature_cols)} total):")
print(feature_cols[:10], '...')

In [ ]:
# Quick table view — what does each row look like?
geochem_gdf[feature_cols[:8]].head()

In [ ]:
# Map of sample locations, lithology background, and known deposits
fig, ax = plt.subplots(figsize=(10, 8))

# Lithology polygons as base layer
h.plot_vector(vector_gdf, ax=ax, title='Sample Locations and Known Deposits',
              alpha=0.3, edgecolor='grey', linewidth=0.4)

# Geochem points coloured by the first element
geochem_gdf.plot(ax=ax, color='steelblue', markersize=15, alpha=0.7,
                 label='Geochem samples')

# Known mineral occurrences as gold stars
tgt_proj = targets_gdf.to_crs(geochem_gdf.crs) if targets_gdf.crs != geochem_gdf.crs else targets_gdf
tgt_proj.plot(ax=ax, marker='*', color='gold', markersize=180,
              edgecolor='black', linewidth=0.8, label='Known deposits', zorder=5)

ax.legend()
plt.tight_layout()
plt.show()

**What you're looking at:** Blue dots are geochemical sample sites. Gold stars are known mineral occurrences. The background polygons are mapped lithology units — different rock types.

**Interpret this map:**
1. Are the sample sites evenly distributed, or do some areas have denser coverage?
2. Do any of the known occurrences appear to cluster in a particular lithology?
3. What does the spatial distribution of samples tell you about potential data bias?

> **Instructor note:** Emphasise that spatial bias in training data is a real problem. If all "background" samples happen to be in one rock type, the model may learn rock type instead of chemistry.

---
## Section 3: PCA — Finding the Main Geochemical Signals

### What is PCA?

Imagine you have 57 measurements per sample. Many of them move together — copper, molybdenum, and gold often co-vary in porphyry systems. PCA finds those co-varying groups and creates new summary variables called **principal components**.

Think of it this way: instead of carrying 57 separate maps, you carry 5 summary maps that together capture most of the information. The first map (PC1) captures the biggest pattern in the whole dataset; the second captures the next biggest independent pattern; and so on.

**What is variance explained?** Each PC explains some fraction of the total spread in the data. If PC1 explains 40% of variance, it means 40% of all the geochemical variation in your dataset can be described by that one new variable.

> **Instructor note:** The coin-sorting analogy works well here. If you have a jar of mixed coins and sort by size first, then colour, you've done something analogous to PCA — finding the most informative axes of variation.

In [ ]:
# SECTION 3: PCA | ~15 min | Target: min 10

# --- YOUR TURN ---
# Should we log-transform the geochemistry before PCA?
# Geochemical data is usually log-normally distributed (a few very high outliers).
# 'log1p' applies log(x+1), which compresses those outliers.
# Try 'none' and compare the variance plot — does the first PC dominate more?
#
TRANSFORM = 'log1p'   # Options: 'log1p'  or  'none'
SCALE = True          # Options: True (standardise) or False (raw log values)

# Safety fallback — don't edit below this line
try:
    assert TRANSFORM in ('log1p', 'none'), "TRANSFORM must be 'log1p' or 'none'"
    assert isinstance(SCALE, bool), "SCALE must be True or False"
except AssertionError as e:
    print(f'Invalid setting: {e}  → using defaults')
    TRANSFORM, SCALE = 'log1p', True

pca_inputs = h.prepare_pca_inputs(
    geochem_gdf, feature_cols,
    transform=TRANSFORM,
    scale_features=SCALE,
)
X_scaled = pca_inputs['X_scaled']
pca_cols  = pca_inputs['pca_cols']
print(f'Transform: {TRANSFORM} | Scale: {SCALE}')
print(f'Feature matrix shape: {X_scaled.shape}')

In [ ]:
# Fit PCA — keep all components so we can see the full variance curve
pca, X_pca = h.fit_pca_model(X_scaled)
print(f'Input dimensions : {X_scaled.shape[1]} elements')
print(f'Output dimensions: {X_pca.shape[1]} principal components')
print(f'Shape of PCA scores: {X_pca.shape}  (one row per sample)')

In [ ]:
# Variance explained plot — how many PCs do we actually need?
fig, ax = h.plot_pca_variance(pca)
plt.show()

**What you're looking at:** Each bar is one principal component. The red line shows cumulative variance — how much total information is captured as you add more PCs. The dashed lines mark 50%, 75%, and 90% thresholds.

**Interpret this plot:**
1. PC1 explains ____% of variance.
2. How many PCs do you need to reach 75% of the total variance? _____
3. Where is the "elbow" in the bar chart — the point where adding another PC stops being useful? _____
4. If you used `TRANSFORM = 'none'`, how does the PC1 percentage change? What does that tell you about the raw geochemical data?

In [ ]:
# --- YOUR TURN ---
# How many PCA components should we plot loadings for?
# A "loading" tells you which elements drive each PC.
# Positive loading = element increases with the PC score.
# Negative loading = element decreases with the PC score.
# Try 2, 3, or 5 — do the later PCs still make geological sense?
#
N_LOAD_COMPONENTS = ___   # Try: 2, 3, or 5

# Safety fallback
try:
    N_LOAD_COMPONENTS = int(N_LOAD_COMPONENTS)
    assert 1 <= N_LOAD_COMPONENTS <= pca.n_components_, \
        f'Must be between 1 and {pca.n_components_}'
except Exception as e:
    print(f'Invalid input: {e}  → using N_LOAD_COMPONENTS = 3')
    N_LOAD_COMPONENTS = 3

fig, axes = h.plot_pca_loadings(pca, pca_cols, n_components=N_LOAD_COMPONENTS)
plt.show()

**What you're looking at:** Each subplot shows one PC. Green bars are elements that load *positively* (high scores point in this direction); orange bars load *negatively*. The length of the bar shows how strongly that element contributes.

**Interpret these loadings:**
1. What element has the strongest positive loading on PC1? _____
2. What element has the strongest negative loading on PC1? _____
3. If you know that gold deposits are associated with arsenic and antimony enrichment, which PC is most likely to separate mineralised from background samples?
4. What geological process might explain the pattern you see in PC2?

> **Instructor note:** If the class has a geochemist, ask them to interpret the loading patterns. Hydrothermal pathfinder elements (As, Sb, Hg, Tl) loading together on one PC is a classic porphyry/epithermal signature.

In [ ]:
# Map the first 3 PCA scores in geographic space
# Do the spatial patterns line up with lithology contacts?
fig, axes = h.plot_spatial_pca_components(
    geochem_gdf, X_pca, pca, n_components=3
)
plt.show()

**What you're looking at:** Each map shows the PC score at each sample location. Warm colours = high score; cool colours = low score. Compare these to the lithology map from Section 2.

**Interpret these maps:**
1. Does PC1 vary smoothly across the study area, or does it show sharp boundaries?
2. Do the PC score boundaries correspond to lithology contacts? What would that mean?
3. Do any known deposit locations (gold stars from the earlier map) appear in high or low PC score zones?

---
## Section 4: K-means Clustering — Geochemical Populations

### What is K-means?

K-means is like sorting a pile of coins into *k* groups. You tell it how many groups you want; it figures out which coins belong together based on similarity. For geochemistry, it groups samples with similar multi-element signatures.

Unlike PCA, K-means gives each sample a **discrete label** (Cluster 0, Cluster 1, ...). That label can then be compared to the geology: do the clusters follow lithology contacts? Do certain clusters correlate with known mineralisation?

**The challenge:** you have to choose *k* before you run the algorithm. We'll use two diagnostics to help:
- **Elbow plot**: inertia (total within-cluster variance) vs k. Look for the "elbow" where the curve flattens.
- **Silhouette score**: measures how well-separated the clusters are (higher = better; max = 1).

> **Instructor note:** Emphasise that there is no single "correct" k. The goal is geological interpretability, not optimising a number.

In [ ]:
# SECTION 4: K-MEANS | ~13 min | Target: min 10

# Run diagnostics for k = 2 through 8 (this takes ~10 seconds)
K_RANGE = range(2, 9)
print('Running K-means diagnostics for k =', list(K_RANGE), '...')
inertias, silhouettes = h.run_kmeans_diagnostics(X_scaled, K_RANGE)
print('Done.')

In [ ]:
# Plot elbow and silhouette curves
fig, axes = h.plot_elbow_silhouette(list(K_RANGE), inertias, silhouettes)
plt.show()

**What you're looking at:** Left = elbow plot (lower inertia = tighter clusters, but always improves with more k). Right = silhouette scores (higher = more distinct clusters; values between 0.2–0.5 are typical for real geochemistry data).

**Interpret these plots:**
1. Where is the elbow in the left plot? Estimate k: _____
2. Which k gives the highest silhouette score? _____
3. Do the two diagnostics agree? If not, which would you trust more, and why?
4. What would it mean geologically to have k = 2 vs k = 6?

In [ ]:
# --- YOUR TURN ---
# Choose the number of clusters based on the plots above.
# The algorithm will suggest an elbow-based value, but you can override it.
# Try at least two different values and compare the maps that follow.
#
MY_K = ___   # Try: 3, 4, or 5

# Safety fallback
try:
    MY_K = int(MY_K)
    assert 2 <= MY_K <= 10, 'k must be between 2 and 10'
except Exception as e:
    print(f'Invalid k: {e}  → using k = 4')
    MY_K = 4

chosen_k      = h.choose_k(list(K_RANGE), inertias, user_override=MY_K)
cluster_labels = h.run_kmeans_clustering(X_scaled, n_clusters=chosen_k)
print(f'\nCluster label counts:')
unique, counts = np.unique(cluster_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Cluster {u}: {c} samples')

In [ ]:
# PCA scatter coloured by cluster — do the groups separate cleanly in PC space?
fig, ax = h.plot_kmeans_pca_scatter(X_pca, cluster_labels)
plt.show()

In [ ]:
# Geographic overlay — do clusters follow lithology contacts?
fig, ax = h.plot_clusters_on_lithology(vector_gdf, geochem_gdf, cluster_labels)
plt.show()

**What you're looking at:** Clusters are shown as coloured points overlaid on lithology polygons. If geochemical clusters align with lithology contacts, that suggests the element patterns are controlled by source rock. If clusters cross lithology boundaries, something else is driving them — perhaps hydrothermal alteration.

**Interpret these maps:**
1. Do any clusters appear to follow lithology contact lines? _____
2. Are any clusters spatially scattered across multiple lithologies? What might that indicate?
3. Are known deposit locations (from Section 2) concentrated in one cluster?
4. Try a different MY_K — do the boundaries shift dramatically, or do the same groups persist?

> **Instructor note:** If clusters track lithology, the geochemistry reflects source rock composition ("lithogeochemistry"). If they cut across contacts, it may reflect hydrothermal overprinting — which is exactly what you're looking for in an exploration context.

---
## Section 5: Supervised ML — Predicting Prospectivity

### From patterns to predictions

PCA and K-means found structure in the data without any labels. Now we'll give the algorithm something specific to learn: **known mineral occurrences**.

We label each geochem sample as:
- **1 (positive)** — within 500 m of a known deposit
- **0 (background)** — everything else

Then we train a **Random Forest**: an ensemble of many decision trees, each trained on a random subset of the data. Each tree votes on whether a new sample looks like a deposit or not. The final output is a probability — how likely is this location to host mineralisation?

```
Geochem samples  ──►  Feature matrix  ──►  Random Forest  ──►  Probability map
(57 elements)         (standardised)       (100 trees)         (0 = low, 1 = high)
```

**Important caveat:** The map at the end of this section was generated by the company's full pipeline, which adds spectral and geophysical data on top of geochemistry. What you're training here is the *same method* — just with fewer input features.

> **Instructor note:** Positive-unlabelled (PU) learning is standard in exploration ML. True negatives don't exist — we just haven't drilled the background samples yet. `class_weight='balanced'` compensates for the imbalanced labels.

In [ ]:
# SECTION 5: SUPERVISED ML | ~20 min | Target: min 10

# Label each geochem sample: 1 = within 500m of a known deposit, 0 = background
y_labels = h.prepare_ml_labels(geochem_gdf, targets_gdf, radius_m=500)

# Show the class balance
n_pos = y_labels.sum()
n_neg = len(y_labels) - n_pos
print(f'\nPositive : {n_pos}  ({100*n_pos/len(y_labels):.1f}%)')
print(f'Background: {n_neg}  ({100*n_neg/len(y_labels):.1f}%)')
print('\nNote: strong imbalance is normal — deposits are rare.')
print("We'll use class_weight='balanced' to handle this.")

# Quick map of labels
fig, ax = plt.subplots(figsize=(10, 7))
h.plot_vector(vector_gdf, ax=ax, alpha=0.2, edgecolor='grey', linewidth=0.4)

bg_mask  = y_labels == 0
pos_mask = y_labels == 1
geochem_gdf[bg_mask].plot(ax=ax,  color='steelblue', markersize=12, alpha=0.6, label='Background')
geochem_gdf[pos_mask].plot(ax=ax, color='red',       markersize=25, alpha=0.9, label='Near deposit (positive)')
tgt_proj.plot(ax=ax, marker='*', color='gold', markersize=180,
              edgecolor='black', linewidth=0.8, label='Known deposits', zorder=5)
ax.set_title('Training Labels')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature matrix for supervised ML — using the same scaled geochem data
# In the full pipeline, spectral and geophysical rasters are added here.
# Here we use geochem only to keep it interpretable.
X_ml = X_scaled.copy()
print(f'Feature matrix: {X_ml.shape[0]} samples × {X_ml.shape[1]} features')

In [ ]:
# --- YOUR TURN ---
# Choose settings for the Random Forest.
# N_ESTIMATORS = number of decision trees. More = slower but often better.
# MAX_DEPTH    = how deep each tree can grow. None = unlimited (can overfit).
#
# Uncomment one option for each, then re-run and compare the ROC curve AUC.

N_ESTIMATORS = 100    # options:  50 (fast)  |  100 (balanced)  |  200 (thorough)
# N_ESTIMATORS = 50
# N_ESTIMATORS = 200

MAX_DEPTH = None      # options:  None (full depth)  |  5 (shallow)  |  10 (medium)
# MAX_DEPTH = 5
# MAX_DEPTH = 10

# Safety fallback
try:
    N_ESTIMATORS = int(N_ESTIMATORS)
    assert 10 <= N_ESTIMATORS <= 500, 'N_ESTIMATORS must be between 10 and 500'
    if MAX_DEPTH is not None:
        MAX_DEPTH = int(MAX_DEPTH)
        assert 1 <= MAX_DEPTH <= 50, 'MAX_DEPTH must be between 1 and 50'
except Exception as e:
    print(f'Invalid setting: {e}  → using defaults')
    N_ESTIMATORS, MAX_DEPTH = 100, None

print(f'Will train: {N_ESTIMATORS} trees, max depth = {MAX_DEPTH}')

In [ ]:
# Train and evaluate the Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

# Hold out 25% of samples for testing
X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_labels, test_size=0.25, random_state=42, stratify=y_labels
)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

# Fit the model
rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

# Probability of belonging to class 1 (deposit-like)
y_prob = rf.predict_proba(X_test)[:, 1]
print(f'\nModel trained. Predicted probability range: {y_prob.min():.3f} – {y_prob.max():.3f}')

# Feature importance table
importance_df = pd.DataFrame({
    'feature'   : pca_cols,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False)

In [ ]:
# ROC and Precision-Recall curves
fig, axes = h.plot_roc_pr_curves(y_test, y_prob)
plt.show()

**What you're looking at:** The ROC curve shows the trade-off between catching real deposits (True Positive Rate) and generating false alarms (False Positive Rate) as you vary the decision threshold. The area under the curve (AUC) is the headline number.

**Interpret these curves:**
1. What is the AUC of your model? _____
2. What would AUC = 0.5 mean? What would AUC = 1.0 mean?
3. In mineral exploration, false negatives (missing a real deposit) are usually more costly than false positives (drilling a dud). Does that affect where you'd set the threshold?
4. Try `N_ESTIMATORS = 50` vs `200`. Does AUC change? Why or why not?

> **Instructor note:** AUC values of 0.65–0.80 on geochemistry-only data are realistic. The full pipeline (geochem + spectral + geophysics) typically reaches 0.80–0.92 because it adds spatial context the point data can't capture.

In [ ]:
# --- YOUR TURN ---
# How many top features do you want to display?
# The bar chart shows which elements the Random Forest found most informative.
# A larger number shows more elements but can get crowded.
#
TOP_N = ___   # Try: 10, 15, or 20

# Safety fallback
try:
    TOP_N = int(TOP_N)
    assert 5 <= TOP_N <= len(pca_cols), f'Must be between 5 and {len(pca_cols)}'
except Exception as e:
    print(f'Invalid input: {e}  → using TOP_N = 15')
    TOP_N = 15

fig, ax = h.plot_feature_importance(importance_df, top_n=TOP_N)
plt.show()

**What you're looking at:** Feature importance shows which geochemical elements contributed most to the model's decisions. Longer bars = more important. These are not the same as PCA loadings — they reflect how much each element *alone* improved the model's predictions.

**Interpret this chart:**
1. What are the top 3 most important elements? _____
2. Are any of these classic gold pathfinders (Au, As, Sb, Hg, Tl, Te)?
3. Are any of the top elements the same ones that loaded strongly on PC1 in Section 3? What does that suggest?
4. If you set MAX_DEPTH = 5, does the importance ranking change significantly?

In [ ]:
# Load and display the pre-generated prospectivity map
# This was produced by the full pipeline: geochem + spectral + geophysics,
# applied to every 150m pixel across the study area — not just sample locations.
# The method is the same as what you just trained.

import rasterio

with rasterio.open(PROSPECTIVITY_TIF) as src:
    prob_map = src.read(1).astype(float)
    prob_map[prob_map == src.nodata] = np.nan
    extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

print(f'Prospectivity map shape: {prob_map.shape}')
print(f'Value range: {np.nanmin(prob_map):.3f} – {np.nanmax(prob_map):.3f}')

fig, axes = h.plot_prospectivity_map(prob_map, extent=extent, threshold=0.5)
plt.suptitle(
    'Pre-generated Prospectivity Map\n'
    '(full pipeline: geochem + spectral + geophysics)',
    fontsize=11
)
plt.tight_layout()
plt.show()

print()
print('This map was produced by a full pipeline (geochem + spectral + geophysics)')
print('applied to every pixel. What you just trained is the same method —')
print('with more features and more samples.')

**What you're looking at:** Left panel = continuous probability (0 = low prospectivity, 1 = high). Right panel = binary classification at a 0.5 threshold. Green = prospective; red = background.

**Interpret this map:**
1. Do the high-probability zones (green) overlap with the known deposits?
2. Are there green zones without known deposits? Are these errors, or candidates for new drilling?
3. How does the spatial pattern compare to the cluster map you made in Section 4?
4. The threshold is currently 0.5. Would you want to lower it (catch more deposits, more false alarms) or raise it (fewer false alarms, risk missing deposits)?

> **Instructor note:** This is the product delivered to the exploration company. The question of where to drill is ultimately a business decision that balances probability of success against drill cost. The map doesn't make that decision — it informs it.

---
## Section 6: Wrap-Up

### What you covered today

| Section | Method | What it found |
|---------|--------|---------------|
| 3 | PCA | Main axes of geochemical variation; element associations |
| 4 | K-means | Geochemical populations; comparison to lithology |
| 5 | Random Forest | Probability of mineralisation at each sample site |

### How this scales up

The notebook trained on point data (one row per sample site). In production:

- Spectral and geophysical rasters add **spatial context** at every pixel
- A spatial cross-validation scheme prevents the model from cheating by memorising nearby deposit locations
- Multiple independent models are trained and stacked — the final map is a consensus
- Uncertainty is quantified and delivered alongside the probability map

The core ML is the same. The complexity lives in the data pipeline.

### What to explore next

- **Geochemical anomaly detection** using Isolation Forest (unsupervised, no deposits needed)
- **Spectral alteration mapping** from remote sensing indices
- **Positive-unlabelled learning** — formally handling the fact that "background" ≠ "no deposit"
- **Spatial cross-validation** — why random train/test split over-estimates performance for spatially autocorrelated data

> **Instructor note:** For questions about the MinersAI platform or to see the full pipeline in action, the contact is on the course materials page.